<a href="https://colab.research.google.com/github/Farshid-997/Alyals-Frontend/blob/main/Audience%20Clustarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
from joblib import Parallel, delayed
from collections import Counter
import logging
import json
from typing import Dict, Any, List

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def evaluate_clusters(features, labels, method_name=""):
    """
    Evaluate clustering results using internal validation metrics

    Args:
        features: Feature matrix used for clustering
        labels: Cluster labels assigned to each sample
        method_name: Name of the clustering method

    Returns:
        Dictionary of evaluation metrics
    """
    # Convert sparse matrix to dense if needed
    if not isinstance(features, np.ndarray):
        features_array = features.toarray()
    else:
        features_array = features

    # Calculate internal validation metrics
    metrics = {}

    try:
        # Silhouette Score (higher is better, range: -1 to 1)
        # Measures how similar an object is to its own cluster compared to other clusters
        silhouette = silhouette_score(features_array, labels)
        metrics['silhouette_score'] = silhouette

        # Calinski-Harabasz Index (higher is better)
        # Ratio of between-cluster dispersion to within-cluster dispersion
        calinski = calinski_harabasz_score(features_array, labels)
        metrics['calinski_harabasz_score'] = calinski

        # Davies-Bouldin Index (lower is better)
        # Average similarity between clusters
        davies = davies_bouldin_score(features_array, labels)
        metrics['davies_bouldin_score'] = davies

        # Display the metrics
        print(f"\n{method_name} Clustering Evaluation:")
        print(f"  Silhouette Score: {silhouette:.4f} [Higher is better, range: -1 to 1]")
        print(f"  Calinski-Harabasz Index: {calinski:.4f} [Higher is better]")
        print(f"  Davies-Bouldin Index: {davies:.4f} [Lower is better]")

    except Exception as e:
        logger.error(f"Error calculating metrics for {method_name}: {str(e)}")

    return metrics

def compare_clustering_methods(features, kmeans_labels, agg_labels, gmm_labels, ensemble_labels):
    """
    Compare different clustering methods using internal validation metrics
    """
    # Convert to array if needed
    if not isinstance(features, np.ndarray):
        features_array = features.toarray()
    else:
        features_array = features

    # Evaluate each method
    kmeans_metrics = evaluate_clusters(features_array, kmeans_labels, "K-Means")
    agg_metrics = evaluate_clusters(features_array, agg_labels, "Agglomerative")
    gmm_metrics = evaluate_clusters(features_array, gmm_labels, "GMM")
    ensemble_metrics = evaluate_clusters(features_array, ensemble_labels, "Ensemble")

    # Combine results for comparison
    results = pd.DataFrame({
        "K-Means": kmeans_metrics,
        "Agglomerative": agg_metrics,
        "GMM": gmm_metrics,
        "Ensemble": ensemble_metrics
    })

    # Display comparison table
    print("\nClustering Methods Comparison:")
    print(results)

    # Find the best method based on each metric
    best_silhouette = results.loc['silhouette_score'].idxmax()
    best_calinski = results.loc['calinski_harabasz_score'].idxmax()
    best_davies = results.loc['davies_bouldin_score'].idxmin()  # Lower is better

    print("\nBest Method Per Metric:")
    print(f"  Silhouette Score: {best_silhouette}")
    print(f"  Calinski-Harabasz Index: {best_calinski}")
    print(f"  Davies-Bouldin Index: {best_davies}")

    return results

def evaluate_consensus(kmeans_labels, agg_labels, gmm_labels, ensemble_labels):
    """
    Evaluate how much the different methods agree with each other and with the ensemble
    """
    n_samples = len(kmeans_labels)
    results = {}

    # Calculate agreement between pairs of methods using Adjusted Rand Index
    # ARI measures the similarity between two clusterings
    # Range: -0.5 to 1.0, where 1.0 is perfect agreement
    results['KMeans vs Agglomerative'] = adjusted_rand_score(kmeans_labels, agg_labels)
    results['KMeans vs GMM'] = adjusted_rand_score(kmeans_labels, gmm_labels)
    results['Agglomerative vs GMM'] = adjusted_rand_score(agg_labels, gmm_labels)

    # Calculate how much each method agrees with the ensemble
    results['KMeans vs Ensemble'] = adjusted_rand_score(kmeans_labels, ensemble_labels)
    results['Agglomerative vs Ensemble'] = adjusted_rand_score(agg_labels, ensemble_labels)
    results['GMM vs Ensemble'] = adjusted_rand_score(gmm_labels, ensemble_labels)

    # Calculate additional agreement metrics using NMI
    # NMI (Normalized Mutual Information) measures the mutual information between two clusterings
    results['NMI: KMeans vs Ensemble'] = normalized_mutual_info_score(kmeans_labels, ensemble_labels)
    results['NMI: Agglomerative vs Ensemble'] = normalized_mutual_info_score(agg_labels, ensemble_labels)
    results['NMI: GMM vs Ensemble'] = normalized_mutual_info_score(gmm_labels, ensemble_labels)

    # Display results
    print("\nConsensus Evaluation:")
    print("Method Agreement (Adjusted Rand Index, range: -0.5 to 1.0):")
    print(f"  KMeans vs Agglomerative: {results['KMeans vs Agglomerative']:.4f}")
    print(f"  KMeans vs GMM: {results['KMeans vs GMM']:.4f}")
    print(f"  Agglomerative vs GMM: {results['Agglomerative vs GMM']:.4f}")

    print("\nMethod vs Ensemble Agreement:")
    print(f"  KMeans vs Ensemble: {results['KMeans vs Ensemble']:.4f}")
    print(f"  Agglomerative vs Ensemble: {results['Agglomerative vs Ensemble']:.4f}")
    print(f"  GMM vs Ensemble: {results['GMM vs Ensemble']:.4f}")

    # Calculate consensus score as average agreement with ensemble
    consensus_score = (results['KMeans vs Ensemble'] +
                      results['Agglomerative vs Ensemble'] +
                      results['GMM vs Ensemble']) / 3

    print(f"\nOverall Consensus Score: {consensus_score:.4f}")

    return results

def dynamic_clusters_job(tag_id, job):
    try:
        # Import numpy within the function to ensure it's available
        import numpy as np

        interval = job.get('interval', 10)
        batch_size = 500
        config_id = job.get('config_id')

        payload: Dict[str, Any] = {
            "tag_id": tag_id,
            "interval": interval,
            "batch_size": batch_size
        }

        print("Payload", payload)

        # Initialize Spark
        spark = SparkSession.builder \
            .appName("FrequentShoppersClusterisation") \
            .getOrCreate()

        # Fetch data
        data = fetch_data(spark, payload)

        if data is None:
            logger.error("No data fetched for clustering")
            return

        logger.info("Collect complete data for clustering")

        print("Sample data:")
        data.show(5)

        # Replace None values with empty strings in 'name' and 'category' columns
        data = data.fillna({'name': 'unknown', 'category': 'unknown'})

        # Convert Spark DataFrame to Pandas for scikit-learn processing
        data_pd = data.toPandas()

        # TF-IDF Vectorization
        data_pd.dropna(inplace=True)
        vectorizer = TfidfVectorizer()

        # Combine name and category for product descriptions
        text_for_clustering = data_pd[['name', 'category']].apply(lambda x: ' '.join(x), axis=1)
        features = vectorizer.fit_transform(text_for_clustering)

        # Define number of clusters
        n_clusters = 10
        print("Features shape:", features.shape)

        if features.shape[0] < n_clusters:
            logger.error("Number of samples is less than the number of clusters requested")
            return

        # K-Means
        kmeans_model = KMeans(n_clusters=n_clusters, random_state=42)
        kmeans_labels = kmeans_model.fit_predict(features)

        print("K-means clustering completed")
        # Evaluate K-means clustering
        kmeans_metrics = evaluate_clusters(features, kmeans_labels, "K-Means")

        # Parallel Agglomerative Clustering
        def batch_agglo_clustering(batch):
            agg_model = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
            return agg_model.fit_predict(batch)

        print("Starting batch agglomerative clustering")
        # Split data into batches for parallel processing
        batch_size = 100
        batches = [features[i:i + batch_size].toarray() for i in range(0, features.shape[0], batch_size)]

        # Perform parallel Agglomerative Clustering on each batch
        agg_batch_labels = Parallel(n_jobs=-1, backend='threading')(delayed(batch_agglo_clustering)(batch) for batch in batches)

        # Combine labels from batches
        agg_labels = np.concatenate(agg_batch_labels)
        # Evaluate Agglomerative clustering
        agg_metrics = evaluate_clusters(features, agg_labels, "Agglomerative")

        # Gaussian Mixture Model (GMM)
        print("Starting GMM clustering")
        gmm_model = GaussianMixture(n_components=n_clusters, random_state=42)
        gmm_labels = gmm_model.fit_predict(features.toarray())
        # Evaluate GMM clustering
        gmm_metrics = evaluate_clusters(features, gmm_labels, "GMM")

        # Combine cluster results using ensemble approach
        ensemble_labels = np.zeros(len(data_pd), dtype=int)

        for i in range(len(data_pd)):
            label_counts = Counter()
            label_counts.update([kmeans_labels[i]])
            label_counts.update([agg_labels[i]])
            label_counts.update([gmm_labels[i]])
            ensemble_labels[i] = label_counts.most_common(1)[0][0]

        # Evaluate ensemble clustering
        ensemble_metrics = evaluate_clusters(features, ensemble_labels, "Ensemble")

        # Compare clustering methods and evaluate consensus
        comparison_results = compare_clustering_methods(features, kmeans_labels, agg_labels, gmm_labels, ensemble_labels)
        consensus_results = evaluate_consensus(kmeans_labels, agg_labels, gmm_labels, ensemble_labels)

        # Add cluster labels to the data
        data_pd['kmeans_cluster'] = kmeans_labels
        data_pd['agg_cluster'] = agg_labels
        data_pd['gmm_cluster'] = gmm_labels
        data_pd['ensemble_cluster'] = ensemble_labels

        # Save intermediate result
        data_pd.to_csv('clustered_data_with_evaluation.csv', index=False)

        # Continue with the rest of your clustering job code...
        # Topic Modeling for cluster labeling
        count_vectorizer = CountVectorizer()
        count_vectorizer.fit(text_for_clustering)

        print("Starting cluster description generation")
        # Create a dictionary to store cluster descriptions and top categories
        cluster_descriptions = {}
        cluster_top_categories = {}

        # Perform LDA for each cluster to extract topics
        for cluster in range(n_clusters):
            # Get the boolean series for the current cluster
            cluster_mask = data_pd['ensemble_cluster'] == cluster

            # Ensure the boolean series aligns with the dataframe index
            if not cluster_mask.any():
                continue

            cluster_data = np.array(text_for_clustering)[cluster_mask]
            cluster_features = count_vectorizer.transform(cluster_data)

            lda = LatentDirichletAllocation(n_components=1, random_state=42)
            lda.fit(cluster_features)

            # Get the top words for the topic
            top_words = [count_vectorizer.get_feature_names_out()[i] for i in lda.components_[0].argsort()[-10:]]

            # Combine top words into a meaningful cluster name
            cluster_name = " ".join(top_words)
            cluster_descriptions[cluster] = cluster_name

            # Find the top 5 categories in the cluster
            cluster_categories = data_pd[data_pd['ensemble_cluster'] == cluster]['category']
            top_categories = cluster_categories.value_counts().head(10).index.tolist()
            cluster_top_categories[cluster] = top_categories

        # Map cluster numbers to meaningful names
        data_pd['cluster_name'] = data_pd['ensemble_cluster'].map(cluster_descriptions)

        # Save the final results with all evaluation metrics
        metrics_summary = {
            "KMeans": kmeans_metrics,
            "Agglomerative": agg_metrics,
            "GMM": gmm_metrics,
            "Ensemble": ensemble_metrics,
            "Consensus": consensus_results
        }

        # Save metrics to JSON file
        with open('clustering_metrics.json', 'w') as f:
            json.dump(metrics_summary, f, indent=2)

        # Save the final results dataframe
        data_pd.to_csv("clustered_final_data_with_evaluation.csv", index=False)

        # Print final summary
        print("\n" + "=" * 50)
        print("CLUSTERING EVALUATION SUMMARY")
        print("=" * 50)
        print("Individual Algorithm Performance:")
        for method, metrics in [("K-Means", kmeans_metrics),
                                ("Agglomerative", agg_metrics),
                                ("GMM", gmm_metrics),
                                ("Ensemble", ensemble_metrics)]:
            print(f"  {method}:")
            print(f"    Silhouette Score: {metrics.get('silhouette_score', 'N/A'):.4f}")
            print(f"    Calinski-Harabasz: {metrics.get('calinski_harabasz_score', 'N/A'):.4f}")
            print(f"    Davies-Bouldin: {metrics.get('davies_bouldin_score', 'N/A'):.4f}")

        print("\nEnsemble Method Improvements:")
        # Calculate improvement percentages
        silhouette_improvement = ((ensemble_metrics.get('silhouette_score', 0) -
                                 max(kmeans_metrics.get('silhouette_score', 0),
                                     agg_metrics.get('silhouette_score', 0),
                                     gmm_metrics.get('silhouette_score', 0))) /
                                 max(kmeans_metrics.get('silhouette_score', 0.001),
                                     agg_metrics.get('silhouette_score', 0.001),
                                     gmm_metrics.get('silhouette_score', 0.001))) * 100

        calinski_improvement = ((ensemble_metrics.get('calinski_harabasz_score', 0) -
                               max(kmeans_metrics.get('calinski_harabasz_score', 0),
                                   agg_metrics.get('calinski_harabasz_score', 0),
                                   gmm_metrics.get('calinski_harabasz_score', 0))) /
                               max(kmeans_metrics.get('calinski_harabasz_score', 0.001),
                                   agg_metrics.get('calinski_harabasz_score', 0.001),
                                   gmm_metrics.get('calinski_harabasz_score', 0.001))) * 100

        davies_improvement = ((min(kmeans_metrics.get('davies_bouldin_score', float('inf')),
                                 agg_metrics.get('davies_bouldin_score', float('inf')),
                                 gmm_metrics.get('davies_bouldin_score', float('inf'))) -
                              ensemble_metrics.get('davies_bouldin_score', 0)) /
                             min(kmeans_metrics.get('davies_bouldin_score', float('inf')),
                                 agg_metrics.get('davies_bouldin_score', float('inf')),
                                 gmm_metrics.get('davies_bouldin_score', float('inf')))) * 100

        print(f"  Silhouette Score: {silhouette_improvement:.2f}% improvement over best individual algorithm")
        print(f"  Calinski-Harabasz: {calinski_improvement:.2f}% improvement over best individual algorithm")
        print(f"  Davies-Bouldin: {davies_improvement:.2f}% improvement over best individual algorithm")

        print("\nConsensus Score:")
        print(f"  Overall Agreement: {consensus_results.get('KMeans vs Ensemble', 0) + consensus_results.get('Agglomerative vs Ensemble', 0) + consensus_results.get('GMM vs Ensemble', 0):.4f}/3")
        print("=" * 50)

        return data_pd

    except Exception as e:
        logger.error(f"Error in dynamic_clusters_job: {str(e)}")
        raise

# Function to fetch data from CSV
def fetch_data(spark, payload):
    """
    Reads product data from the CSV file in Colab
    """
    print(f"Fetching data with payload: {payload}")

    # Path to your CSV file - UPDATED PATH
    csv_path = "/content/sample_data/products (1).csv"

    # Read CSV file into pandas DataFrame first to check structure
    import pandas as pd
    pdf = pd.read_csv(csv_path)

    # Print column names and sample data for verification
    print("CSV columns:", pdf.columns.tolist())
    print("Sample data (first 3 rows):")
    print(pdf.head(3))

    # Check if name and category columns exist, if not, handle appropriately
    required_columns = ['name', 'category']
    missing_columns = [col for col in required_columns if col not in pdf.columns]

    if missing_columns:
        print(f"Warning: Missing required columns: {missing_columns}")
        # If columns are missing but have equivalents with different names, rename them
        # This is an example - modify based on your actual CSV structure
        column_mapping = {}
        if 'name' not in pdf.columns and 'product_name' in pdf.columns:
            column_mapping['product_name'] = 'name'
        if 'category' not in pdf.columns and 'product_category' in pdf.columns:
            column_mapping['product_category'] = 'category'

        if column_mapping:
            pdf = pdf.rename(columns=column_mapping)
        else:
            # If no equivalents exist, create dummy columns with default values
            for col in missing_columns:
                pdf[col] = "unknown"

    # Convert to Spark DataFrame
    df = spark.createDataFrame(pdf)

    # Apply batch size limit if specified
    if 'batch_size' in payload and payload['batch_size'] > 0:
        df = df.limit(payload['batch_size'])

    return df

# Run the clustering job with evaluation
if __name__ == "__main__":
    # Run the clustering
    tag_id = "product_data"
    job = {
        "interval": 10,
        "config_id": "demo_config",
        "batch_size": 1000  # Process up to 1000 products
    }

    result_df = dynamic_clusters_job(tag_id, job)

    # Print final message
    print("\n" + "=" * 50)
    print("CLUSTERING WITH EVALUATION COMPLETE")
    print("=" * 50)
    print("Cluster results saved to 'clustered_final_data_with_evaluation.csv'")
    print("Metrics saved to 'clustering_metrics.json'")
    print("=" * 50)

Payload {'tag_id': 'product_data', 'interval': 10, 'batch_size': 500}
Fetching data with payload: {'tag_id': 'product_data', 'interval': 10, 'batch_size': 500}
CSV columns: ['eventId', 'name', 'email', 'phone', 'category', 'muid', 'event', 'tagId', 'value', 'quantity']
Sample data (first 3 rows):
                    eventId                                 name email phone  \
0  6811e9103bbf55d2d5fead91  【RE′DE′STU（リ・デ・スタ）】フォトリアル醤油皿-enman-   NaN   NaN   
1  6811e9103bbf55d2d5fead91  【RE′DE′STU（リ・デ・スタ）】フォトリアル醤油皿-enman-   NaN   NaN   
2  6811e9103bbf55d2d5fead91  【RE′DE′STU（リ・デ・スタ）】フォトリアル醤油皿-enman-   NaN   NaN   

                                            category  \
0  その他\tララいいもの広島\tララいいもの広島　その他\t1,101円～2,200円\t広島...   
1  その他\tララいいもの広島\tララいいもの広島　その他\t1,101円～2,200円\t広島...   
2  その他\tララいいもの広島\tララいいもの広島　その他\t1,101円～2,200円\t広島...   

                                   muid     event  \
0  268572c2-7800-44ee-b48d-65a9fe1beac6  ViewItem   
1  268572c2-7800-44ee-b48d-65a9fe1beac6  ViewItem 

# New Section